# SHNU-LLM v0.1 — a decoder-only Transformer trained **from scratch**

*No pretrained model weights are used to initialize SHNU-LLM.* Every weight in this
notebook is randomly initialized and trained through our own training loop on openly
licensed text. We use open-source **software** (PyTorch, HuggingFace `tokenizers`,
HuggingFace `datasets`) but **no pretrained language-model checkpoint of any kind**.

The architecture (RoPE, RMSNorm, SwiGLU, causal self-attention), the tokenizer, the
training process, and the final weights are ours.

**Run order:** top to bottom. Set `PRESET` in Section 2. `Runtime → Change runtime type → T4 GPU`
(free) is recommended. Checkpoints resume automatically if the runtime disconnects.


## 1. Environment

In [ ]:
# Install the only two extra libraries we need (software only — no pretrained weights).
%pip install -q tokenizers datasets 2>/dev/null

import math, os, json, time, random, platform
from dataclasses import dataclass, asdict, field
from typing import Optional
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def env_report(device=DEVICE):
    rep = {"python": platform.python_version(), "torch": torch.__version__,
           "cuda_available": torch.cuda.is_available(), "device": device}
    if torch.cuda.is_available():
        rep["gpu_name"] = torch.cuda.get_device_name(0)
        rep["gpu_memory_gb"] = round(torch.cuda.get_device_properties(0).total_memory/1e9, 2)
    return rep

print(json.dumps(env_report(), indent=2))
if DEVICE == "cpu":
    print("\n[!] No GPU detected. Runtime -> Change runtime type -> T4 GPU (free) is recommended.")

## 2. Configuration
All important knobs live in one `ShnuConfig` object. `PRESET` picks a fast smoke config or the ~34M-param T4 config.

In [ ]:
@dataclass
class ShnuConfig:
    # --- identity ---
    name: str = "SHNU-LLM"
    version: str = "v0.1"

    # --- tokenizer ---
    vocab_size: int = 16000          # trained BPE vocab (incl. special tokens)

    # --- model architecture ---
    n_layers: int = 8
    n_heads: int = 8
    n_kv_heads: Optional[int] = None  # None => == n_heads (full MHA); else GQA
    d_model: int = 512
    d_ff: Optional[int] = None        # None => derived SwiGLU width
    block_size: int = 256             # context length (max sequence length)
    rope_theta: float = 10000.0
    dropout: float = 0.0
    tie_embeddings: bool = True

    # --- training ---
    batch_size: int = 32              # micro-batch (per optimizer step slice)
    grad_accum_steps: int = 1
    max_steps: int = 2000
    warmup_steps: int = 100
    lr: float = 3e-4
    min_lr_ratio: float = 0.1         # cosine floor = lr * ratio
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95
    grad_clip: float = 1.0
    eval_interval: int = 200
    eval_iters: int = 50
    sample_interval: int = 500
    checkpoint_interval: int = 500
    log_interval: int = 20
    seed: int = 1337

    # --- runtime / io ---
    out_dir: str = "SHNU_LLM"
    dtype: str = "auto"               # "auto"|"float32"|"bfloat16"|"float16"
    compile_model: bool = False

    def __post_init__(self):
        assert self.d_model % self.n_heads == 0, "d_model must divide n_heads"
        if self.n_kv_heads is None:
            self.n_kv_heads = self.n_heads
        assert self.n_heads % self.n_kv_heads == 0, "n_heads must divide n_kv_heads"
        if self.d_ff is None:
            # SwiGLU: keep param-equivalent to 4*d_model MLP => ~ (8/3)*d_model,
            # rounded to a multiple of 64.
            hidden = int(8 / 3 * self.d_model)
            self.d_ff = ((hidden + 63) // 64) * 64

    @property
    def head_dim(self) -> int:
        return self.d_model // self.n_heads

    def to_json(self) -> str:
        return json.dumps(asdict(self), indent=2)

In [ ]:
# ---- choose a preset ----
PRESET = "t4"   # "smoke" (fast self-test, ~1 min) or "t4" (~34M params, free T4 GPU)

if PRESET == "smoke":
    cfg = ShnuConfig(vocab_size=4000, n_layers=4, n_heads=4, d_model=128, block_size=128,
                      batch_size=16, grad_accum_steps=1, max_steps=300, warmup_steps=30,
                      lr=6e-4, eval_interval=100, eval_iters=20, sample_interval=150,
                      checkpoint_interval=150, log_interval=50, out_dir="SHNU_LLM", seed=1337)
    DATA_MAX_CHARS = 2_000_000
else:  # t4  (~34M params)
    cfg = ShnuConfig(vocab_size=16000, n_layers=8, n_heads=8, d_model=512, block_size=512,
                      batch_size=16, grad_accum_steps=4, max_steps=4000, warmup_steps=200,
                      lr=3e-4, eval_interval=250, eval_iters=100, sample_interval=1000,
                      checkpoint_interval=500, log_interval=25, out_dir="SHNU_LLM", seed=1337)
    DATA_MAX_CHARS = 60_000_000   # subset of WikiText-103; scale up in later versions

print(cfg.to_json())

## 3. Reproducibility
Seed everything and record the full environment/config.

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def resolve_dtype(cfg: ShnuConfig, device: str) -> torch.dtype:
    if cfg.dtype == "float32":
        return torch.float32
    if cfg.dtype == "bfloat16":
        return torch.bfloat16
    if cfg.dtype == "float16":
        return torch.float16
    # auto
    if device == "cuda":
        if torch.cuda.is_bf16_supported():
            return torch.bfloat16
        return torch.float16
    return torch.float32

In [ ]:
set_seed(cfg.seed)
RUN_META = {"env": env_report(), "config": asdict(cfg), "preset": PRESET}
print(json.dumps(RUN_META, indent=2))

## 4. Dataset acquisition
**WikiText-103** (human-written Wikipedia text, license **CC BY-SA 3.0**) via HuggingFace `datasets`. We take a subset that is realistic for free compute; the pipeline scales to the full set later. Falls back to **Tiny Shakespeare** (public domain) if the hub is unreachable. We do **not** claim ownership of this third-party text.

In [ ]:
DATASET_INFO = {}
raw_records = []
try:
    from datasets import load_dataset
    ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train", streaming=True)
    buf, total = [], 0
    for ex in ds:
        t = ex["text"]
        if t and t.strip():
            buf.append(t); total += len(t)
        if total >= DATA_MAX_CHARS:
            break
    raw_records = buf
    DATASET_INFO = {"source": "wikitext-103-raw-v1", "license": "CC BY-SA 3.0",
                    "chars": total, "records": len(raw_records)}
except Exception as e:
    print("[fallback] datasets hub unavailable:", repr(e))
    import urllib.request
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    txt = urllib.request.urlopen(url, timeout=60).read().decode("utf-8")
    raw_records = txt.split("\n\n")
    DATASET_INFO = {"source": "tiny-shakespeare", "license": "public domain",
                    "chars": len(txt), "records": len(raw_records)}
print(json.dumps(DATASET_INFO, indent=2))
print("example record:", repr(raw_records[len(raw_records)//2][:200]))

## 5. Dataset cleaning
Remove empty/short/duplicate records and normalize text.

In [ ]:
def clean_text(text: str) -> str:
    """Light normalization suitable for LM pretraining."""
    # normalize newlines, strip carriage returns and null bytes
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\x00", "")
    return text


def clean_records(records, min_chars: int = 1):
    """Filter/clean an iterable of raw text records.

    Removes empty / whitespace-only / too-short records and exact duplicates.
    Returns (kept_list, stats_dict).
    """
    seen = set()
    kept = []
    n_in = n_empty = n_short = n_dup = 0
    for r in records:
        n_in += 1
        if not r or not r.strip():
            n_empty += 1
            continue
        r = clean_text(r)
        if len(r.strip()) < min_chars:
            n_short += 1
            continue
        h = hash(r)
        if h in seen:
            n_dup += 1
            continue
        seen.add(h)
        kept.append(r)
    stats = dict(records_in=n_in, kept=len(kept),
                 removed_empty=n_empty, removed_short=n_short, removed_dup=n_dup)
    return kept, stats

In [ ]:
records, clean_stats = clean_records(raw_records, min_chars=32)
RUN_META["clean_stats"] = clean_stats
print(json.dumps(clean_stats, indent=2))

## 6. Tokenizer
A byte-level **BPE** tokenizer trained from scratch on our corpus (not downloaded from any pretrained model). Byte-level fallback guarantees no true `<unk>`.

In [ ]:
SPECIAL_TOKENS = ["<pad>", "<unk>", "<bos>", "<eos>"]


def train_tokenizer(text_iter, vocab_size: int, save_path: str):
    """Train a byte-level BPE tokenizer from scratch and save it."""
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

    tok = Tokenizer(models.BPE(unk_token="<unk>"))
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=SPECIAL_TOKENS,
        show_progress=False,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    )
    tok.train_from_iterator(text_iter, trainer=trainer)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    tok.save(save_path)
    return tok


def load_tokenizer(path: str):
    from tokenizers import Tokenizer
    return Tokenizer.from_file(path)


def encode(tok, text: str):
    return tok.encode(text).ids


def decode(tok, ids):
    return tok.decode(ids)

In [ ]:
TOK_PATH = os.path.join(cfg.out_dir, "tokenizer", "shnu_bpe.json")
t0 = time.time()
# train on a bounded slice of records for speed; scales up in later versions
tok = train_tokenizer(iter(records[:200000]), cfg.vocab_size, TOK_PATH)
cfg.vocab_size = tok.get_vocab_size()
RUN_META["tokenizer"] = {"vocab_size": cfg.vocab_size, "train_s": round(time.time()-t0,1),
                         "special_tokens": SPECIAL_TOKENS, "path": TOK_PATH}
print(json.dumps(RUN_META["tokenizer"], indent=2))

## 7. Tokenization
Verify TEXT → IDs → TEXT round-trips and measure tokenization length.

In [ ]:
probe = "SHNU-LLM is a small language model trained from scratch."
ids = encode(tok, probe)
back = decode(tok, ids)
lengths = [len(tok.encode(r).ids) for r in records[:500]]
tok_stats = {"roundtrip_ok": back.strip()==probe.strip(),
             "example_ids": ids[:16], "n_ids": len(ids),
             "avg_ids_per_record": round(float(np.mean(lengths)),1),
             "chars_per_token": round(sum(len(r) for r in records[:500])/max(1,sum(lengths)),2)}
RUN_META["tokenization"] = tok_stats
print(json.dumps(tok_stats, indent=2)); print("decoded:", repr(back))

## 8. Dataset packing
Concatenate documents (EOS between), split train/val with **no overlap**.

In [ ]:
def build_token_stream(tok, records, eos_id: int):
    """Encode records and concatenate with an EOS between documents."""
    ids = []
    for r in records:
        ids.extend(tok.encode(r).ids)
        ids.append(eos_id)
    return np.array(ids, dtype=np.uint16 if tok.get_vocab_size() < 65536 else np.uint32)


def train_val_split(token_ids: np.ndarray, val_ratio: float = 0.05):
    """Split the token stream into train/val with NO overlap (no leakage)."""
    n_val = int(len(token_ids) * val_ratio)
    train_ids = token_ids[:-n_val]
    val_ids = token_ids[-n_val:]
    return train_ids, val_ids


class PackedDataset:
    """Serves contiguous (block_size+1) windows for causal LM training."""
    def __init__(self, token_ids: np.ndarray, block_size: int, device: str):
        self.data = torch.from_numpy(token_ids.astype(np.int64))
        self.block_size = block_size
        self.device = device

    def __len__(self):
        return len(self.data) - self.block_size - 1

    def get_batch(self, batch_size: int, generator: torch.Generator):
        ix = torch.randint(len(self), (batch_size,), generator=generator)
        x = torch.stack([self.data[i:i + self.block_size] for i in ix])
        y = torch.stack([self.data[i + 1:i + 1 + self.block_size] for i in ix])
        if self.device == "cuda":
            x = x.pin_memory().to(self.device, non_blocking=True)
            y = y.pin_memory().to(self.device, non_blocking=True)
        else:
            x, y = x.to(self.device), y.to(self.device)
        return x, y

In [ ]:
eos_id = tok.token_to_id("<eos>")
stream = build_token_stream(tok, records, eos_id)
train_ids, val_ids = train_val_split(stream, val_ratio=0.02 if PRESET=="t4" else 0.05)
datasets = {"train": PackedDataset(train_ids, cfg.block_size, DEVICE),
            "val":   PackedDataset(val_ids,   cfg.block_size, DEVICE)}
RUN_META["tokens"] = {"total": int(len(stream)), "train": int(len(train_ids)), "val": int(len(val_ids))}
print(json.dumps(RUN_META["tokens"], indent=2))

## 9. Model architecture
Decoder-only Transformer implemented from scratch: token embeddings, **RoPE** positions, causal self-attention with our own Q/K/V projections, **SwiGLU** FFN, **RMSNorm**, residual connections, tied output projection, and the causal LM loss.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight


def build_rope_cache(block_size: int, head_dim: int, theta: float, device, dtype):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(block_size, device=device).float()
    freqs = torch.outer(t, inv_freq)              # (T, head_dim/2)
    cos = freqs.cos()[None, None, :, :]            # (1,1,T,hd/2)
    sin = freqs.sin()[None, None, :, :]
    return cos.to(dtype), sin.to(dtype)


def apply_rope(x, cos, sin):
    # x: (B, n_heads, T, head_dim)
    T = x.size(2)
    cos, sin = cos[:, :, :T, :], sin[:, :, :T, :]
    x1, x2 = x[..., ::2], x[..., 1::2]
    rx1 = x1 * cos - x2 * sin
    rx2 = x1 * sin + x2 * cos
    out = torch.stack((rx1, rx2), dim=-1).flatten(-2)
    return out


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: ShnuConfig):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.head_dim = cfg.head_dim
        self.q_proj = nn.Linear(cfg.d_model, self.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.n_heads * self.head_dim, cfg.d_model, bias=False)
        self.dropout = cfg.dropout

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        if self.n_kv_heads != self.n_heads:                 # GQA: expand KV heads
            rep = self.n_heads // self.n_kv_heads
            k = k.repeat_interleave(rep, dim=1)
            v = v.repeat_interleave(rep, dim=1)
        # scaled dot-product attention with causal mask (PyTorch primitive)
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True,
            dropout_p=self.dropout if self.training else 0.0,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, self.n_heads * self.head_dim)
        return self.o_proj(y)


class SwiGLU(nn.Module):
    def __init__(self, cfg: ShnuConfig):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.w_up = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.w_down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.drop(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))


class Block(nn.Module):
    def __init__(self, cfg: ShnuConfig):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.attn_norm(x), cos, sin)   # residual
        x = x + self.ffn(self.ffn_norm(x))               # residual
        return x


class ShnuLM(nn.Module):
    def __init__(self, cfg: ShnuConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm_f = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight       # weight tying
        # rope cache (registered as buffer, rebuilt for device/dtype at runtime)
        self._cos = None
        self._sin = None
        self.apply(self._init_weights)
        # scaled init for residual projections (GPT-2 style)
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _rope(self, device, dtype):
        if self._cos is None or self._cos.device != device or self._cos.dtype != dtype:
            self._cos, self._sin = build_rope_cache(
                self.cfg.block_size, self.cfg.head_dim, self.cfg.rope_theta, device, dtype)
        return self._cos, self._sin

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, "sequence longer than block_size"
        x = self.drop(self.tok_emb(idx))
        cos, sin = self._rope(x.device, x.dtype)
        for block in self.blocks:
            x = block(x, cos, sin)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1),
                ignore_index=-100)
        return logits, loss

    # ----- 10. PARAMETER COUNT -----
    def num_params(self, non_embedding: bool = False) -> int:
        n = sum(p.numel() for p in self.parameters())
        if non_embedding and self.cfg.tie_embeddings:
            n -= self.tok_emb.weight.numel()   # counted once due to tying
        return n

    def param_breakdown(self) -> dict:
        d = {}
        d["embedding"] = self.tok_emb.weight.numel()
        d["blocks"] = sum(p.numel() for b in self.blocks for p in b.parameters())
        d["final_norm"] = sum(p.numel() for p in self.norm_f.parameters())
        d["lm_head"] = 0 if self.cfg.tie_embeddings else self.lm_head.weight.numel()
        d["total"] = self.num_params()
        return d

    # ----- 16. GENERATION (our own) -----
    @torch.no_grad()
    def generate(self, idx, max_new_tokens: int, temperature: float = 1.0,
                 top_k: Optional[int] = None, top_p: Optional[float] = None,
                 eos_id: Optional[int] = None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            if temperature <= 0:                        # greedy
                next_id = torch.argmax(logits, dim=-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k is not None:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = -float("inf")
                if top_p is not None:
                    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
                    probs = F.softmax(sorted_logits, dim=-1)
                    cum = torch.cumsum(probs, dim=-1)
                    mask = cum - probs > top_p
                    sorted_logits[mask] = -float("inf")
                    logits = torch.full_like(logits, -float("inf")).scatter(
                        1, sorted_idx, sorted_logits)
                probs = F.softmax(logits, dim=-1)
                next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
            if eos_id is not None and (next_id == eos_id).all():
                break
        return idx

## 10. Parameter count & random initialization
Instantiate the model, print the exact parameter count and a per-part breakdown, and confirm **no checkpoint was loaded**.

In [ ]:
set_seed(cfg.seed)
model = ShnuLM(cfg).to(DEVICE)
RUN_META["params"] = {"total": model.num_params(), "non_embedding": model.num_params(True),
                      "breakdown": model.param_breakdown()}
print(f"TOTAL PARAMS: {model.num_params():,}  (non-embedding: {model.num_params(True):,})")
print("BREAKDOWN:", json.dumps(model.param_breakdown(), indent=2))
print("\nMODEL INITIALIZATION:\n  Pretrained checkpoint: NONE\n  Random initialization: TRUE")
# sanity: initial loss should be ~ ln(vocab_size) for a correctly-initialized model
xb, yb = datasets["train"].get_batch(4, torch.Generator().manual_seed(0))
with torch.no_grad():
    _, l0 = model(xb, yb)
print(f"  initial loss = {l0.item():.4f}  vs  ln(vocab) = {math.log(cfg.vocab_size):.4f}")

## 11. Training (with 12. checkpointing & 13. validation)
Causal next-token prediction, AdamW, cosine schedule with warmup, gradient clipping, gradient accumulation, mixed precision on GPU, periodic validation, periodic sampling, and checkpointing. `resume_from` continues from the latest checkpoint instead of restarting.

In [ ]:
def get_lr(step: int, cfg: ShnuConfig) -> float:
    if step < cfg.warmup_steps:
        return cfg.lr * (step + 1) / max(1, cfg.warmup_steps)
    if step >= cfg.max_steps:
        return cfg.lr * cfg.min_lr_ratio
    ratio = (step - cfg.warmup_steps) / max(1, cfg.max_steps - cfg.warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * ratio))
    return cfg.lr * cfg.min_lr_ratio + coeff * (cfg.lr - cfg.lr * cfg.min_lr_ratio)


def configure_optimizer(model, cfg: ShnuConfig):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.dim() >= 2:
            decay.append(p)
        else:
            no_decay.append(p)
    groups = [
        {"params": decay, "weight_decay": cfg.weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]
    return torch.optim.AdamW(groups, lr=cfg.lr, betas=(cfg.beta1, cfg.beta2))


@torch.no_grad()
def estimate_loss(model, datasets: dict, cfg: ShnuConfig, gen: torch.Generator, ctx):
    model.eval()
    out = {}
    for split, ds in datasets.items():
        losses = torch.zeros(cfg.eval_iters)
        for k in range(cfg.eval_iters):
            x, y = ds.get_batch(cfg.batch_size, gen)
            with ctx:
                _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def save_checkpoint(path, model, optimizer, cfg, step, best_val, log, rng_state=None):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    ckpt = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "config": asdict(cfg),
        "step": step,
        "best_val": best_val,
        "log": log,
        "torch_rng_state": torch.get_rng_state(),
        "numpy_rng_state": np.random.get_state(),
    }
    if rng_state is not None:
        ckpt["cuda_rng_state"] = rng_state
    torch.save(ckpt, path)


def load_checkpoint(path, model, optimizer=None, map_location="cpu"):
    ckpt = torch.load(path, map_location=map_location, weights_only=False)
    model.load_state_dict(ckpt["model"])
    if optimizer is not None and "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    return ckpt


def train(model, datasets, cfg: ShnuConfig, device: str,
          resume_from: Optional[str] = None, verbose: bool = True,
          tokenizer=None, eos_id=None, sample_prompt="\n"):
    os.makedirs(os.path.join(cfg.out_dir, "checkpoints"), exist_ok=True)
    os.makedirs(os.path.join(cfg.out_dir, "samples"), exist_ok=True)
    os.makedirs(os.path.join(cfg.out_dir, "logs"), exist_ok=True)

    model.to(device)
    optimizer = configure_optimizer(model, cfg)
    ptdtype = resolve_dtype(cfg, device)
    use_amp = device == "cuda" and ptdtype in (torch.float16, torch.bfloat16)
    ctx = (torch.autocast(device_type="cuda", dtype=ptdtype) if use_amp
           else torch.autocast(device_type="cpu", dtype=torch.bfloat16)
           if device == "cpu" and ptdtype == torch.bfloat16
           else _nullcontext())
    scaler = torch.amp.GradScaler(enabled=(use_amp and ptdtype == torch.float16))

    start_step, best_val = 0, float("inf")
    log = {"step": [], "train_loss": [], "val_loss": [], "lr": []}
    if resume_from and os.path.exists(resume_from):
        ckpt = load_checkpoint(resume_from, model, optimizer, map_location=device)
        start_step = ckpt.get("step", 0)
        best_val = ckpt.get("best_val", float("inf"))
        log = ckpt.get("log", log)
        if verbose:
            print(f"[resume] from step {start_step} (best_val={best_val:.4f})")

    gen = torch.Generator().manual_seed(cfg.seed + start_step)
    model.train()
    t0 = time.time()
    tokens_per_step = cfg.batch_size * cfg.grad_accum_steps * cfg.block_size
    running = None

    for step in range(start_step, cfg.max_steps):
        lr = get_lr(step, cfg)
        for pg in optimizer.param_groups:
            pg["lr"] = lr

        optimizer.zero_grad(set_to_none=True)
        loss_accum = 0.0
        for _ in range(cfg.grad_accum_steps):
            x, y = datasets["train"].get_batch(cfg.batch_size, gen)
            with ctx:
                _, loss = model(x, y)
                loss = loss / cfg.grad_accum_steps
            scaler.scale(loss).backward()
            loss_accum += loss.item()
        if cfg.grad_clip > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()

        running = loss_accum if running is None else 0.9 * running + 0.1 * loss_accum
        if verbose and step % cfg.log_interval == 0:
            dt = time.time() - t0
            tps = tokens_per_step * (step - start_step + 1) / max(dt, 1e-9)
            mem = (torch.cuda.max_memory_allocated() / 1e9) if device == "cuda" else 0.0
            print(f"step {step:5d} | loss {loss_accum:.4f} | ema {running:.4f} "
                  f"| lr {lr:.2e} | {tps:8.0f} tok/s | {mem:.2f}GB")

        # periodic validation
        if step > 0 and step % cfg.eval_interval == 0:
            metrics = estimate_loss(model, datasets, cfg, gen, ctx)
            log["step"].append(step)
            log["train_loss"].append(metrics["train"])
            log["val_loss"].append(metrics["val"])
            log["lr"].append(lr)
            if verbose:
                print(f"  >> eval step {step}: train {metrics['train']:.4f} "
                      f"val {metrics['val']:.4f} ppl {math.exp(metrics['val']):.2f}")
            if metrics["val"] < best_val:
                best_val = metrics["val"]
                save_checkpoint(os.path.join(cfg.out_dir, "checkpoints", "checkpoint_best.pt"),
                                model, optimizer, cfg, step, best_val, log)

        # periodic sampling
        if tokenizer is not None and step > 0 and step % cfg.sample_interval == 0:
            sample = sample_text(model, tokenizer, cfg, device, sample_prompt,
                                 max_new_tokens=120, temperature=0.8, top_k=50, eos_id=eos_id)
            with open(os.path.join(cfg.out_dir, "samples", f"step_{step}.txt"), "w") as f:
                f.write(sample)
            if verbose:
                print(f"  >> sample @ {step}: {sample[:120]!r}")

        # periodic checkpoint
        if step > 0 and step % cfg.checkpoint_interval == 0:
            save_checkpoint(os.path.join(cfg.out_dir, "checkpoints", f"checkpoint_step_{step:06d}.pt"),
                            model, optimizer, cfg, step, best_val, log)
            save_checkpoint(os.path.join(cfg.out_dir, "checkpoints", "checkpoint_latest.pt"),
                            model, optimizer, cfg, step, best_val, log)

    # final eval + checkpoint
    metrics = estimate_loss(model, datasets, cfg, gen, ctx)
    save_checkpoint(os.path.join(cfg.out_dir, "checkpoints", "checkpoint_latest.pt"),
                    model, optimizer, cfg, cfg.max_steps, best_val, log)
    with open(os.path.join(cfg.out_dir, "logs", "train_log.json"), "w") as f:
        json.dump(log, f, indent=2)
    total_time = time.time() - t0
    return {
        "final_train_loss": metrics["train"],
        "final_val_loss": metrics["val"],
        "final_val_ppl": math.exp(metrics["val"]),
        "best_val_loss": best_val,
        "steps": cfg.max_steps,
        "train_time_s": total_time,
        "tokens_seen": tokens_per_step * (cfg.max_steps - start_step),
        "log": log,
    }


class _nullcontext:
    def __enter__(self): return None
    def __exit__(self, *a): return False

def sample_text(model, tokenizer, cfg, device, prompt, max_new_tokens=100,
                temperature=0.8, top_k=50, top_p=None, eos_id=None):
    ids = tokenizer.encode(prompt).ids
    if len(ids) == 0:
        ids = [tokenizer.token_to_id("<bos>")]
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    out = model.generate(idx, max_new_tokens=max_new_tokens, temperature=temperature,
                         top_k=top_k, top_p=top_p, eos_id=eos_id)
    return tokenizer.decode(out[0].tolist())

## 12. Checkpoint storage (optional Google Drive)
By default checkpoints go to the Colab local disk under `SHNU_LLM/checkpoints/` (lost when the runtime is recycled). **To persist across disconnects, mount Google Drive** — this opens a Google authorization popup you approve yourself; the notebook cannot approve it for you.

In [ ]:
USE_DRIVE = False   # set True and run to persist checkpoints in Google Drive (requires your approval)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")            # <-- you approve the OAuth popup
    cfg.out_dir = "/content/drive/MyDrive/SHNU_LLM"
os.makedirs(os.path.join(cfg.out_dir, "checkpoints"), exist_ok=True)
print("Checkpoints ->", os.path.abspath(os.path.join(cfg.out_dir, "checkpoints")))

### Run pretraining
Auto-resumes from `checkpoint_latest.pt` if present. Re-run this cell after a disconnect to continue.

In [ ]:
latest = os.path.join(cfg.out_dir, "checkpoints", "checkpoint_latest.pt")
resume = latest if os.path.exists(latest) else None
train_result = train(model, datasets, cfg, DEVICE, resume_from=resume,
                     tokenizer=tok, eos_id=eos_id,
                     sample_prompt="The " if PRESET=="t4" else "First Citizen:\n", verbose=True)
RUN_META["train_result"] = {k: v for k, v in train_result.items() if k != "log"}
print(json.dumps(RUN_META["train_result"], indent=2))

## 14. Evaluation
Validation loss and perplexity from held-out data, plus a training-curve plot.

In [ ]:
log = train_result["log"]
if log["step"]:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6,4))
    plt.plot(log["step"], log["train_loss"], label="train")
    plt.plot(log["step"], log["val_loss"], label="val")
    plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.title("SHNU-LLM training"); plt.grid(True)
    plt.savefig(os.path.join(cfg.out_dir, "logs", "loss_curve.png"), dpi=120, bbox_inches="tight")
    plt.show()
val = train_result["final_val_loss"]
print(f"final val loss = {val:.4f} | perplexity = {math.exp(val):.2f}")

## 15. Text generation
Our own sampler: greedy, temperature, top-k, top-p. All text comes from SHNU-LLM — no other model is involved.

In [ ]:
def sample_text(model, tokenizer, cfg, device, prompt, max_new_tokens=100,
                temperature=0.8, top_k=50, top_p=None, eos_id=None):
    ids = tokenizer.encode(prompt).ids
    if len(ids) == 0:
        ids = [tokenizer.token_to_id("<bos>")]
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    out = model.generate(idx, max_new_tokens=max_new_tokens, temperature=temperature,
                         top_k=top_k, top_p=top_p, eos_id=eos_id)
    return tokenizer.decode(out[0].tolist())

In [ ]:
prompts = ["The ", "In the beginning ", "She looked at the "] if PRESET=="t4" \
          else ["First Citizen:\n", "KING RICHARD:\n"]
for p in prompts:
    print("PROMPT:", repr(p))
    print("  greedy :", repr(sample_text(model, tok, cfg, DEVICE, p, 60, temperature=0.0, eos_id=eos_id)))
    print("  top-k  :", repr(sample_text(model, tok, cfg, DEVICE, p, 60, temperature=0.8, top_k=50, eos_id=eos_id)))
    print("  top-p  :", repr(sample_text(model, tok, cfg, DEVICE, p, 60, temperature=0.9, top_p=0.95, eos_id=eos_id)))

## 16. Instruction-tuning preparation (separate from pretraining)
This only **prepares** the SFT format for a future **SHNU-LLM INSTRUCT**. The base model above remains the deliverable; we do not fine-tune here.

In [ ]:
SFT_TEMPLATE = "<bos>### Instruction:\n{instruction}\n\n### Response:\n{response}<eos>"
def format_sft(instruction, response):
    return SFT_TEMPLATE.format(instruction=instruction, response=response)
example = format_sft("Summarize what SHNU-LLM is.",
                     "A small decoder-only Transformer trained from scratch.")
print(example)
print("SFT example token length:", len(tok.encode(example).ids))
print("\n[note] Base pretraining and instruction tuning are kept strictly separate.")

## 17. Final model export
Save weights + config + tokenizer into a clean directory that can be reloaded later without retraining.

In [ ]:
final_dir = os.path.join(cfg.out_dir, "final")
os.makedirs(final_dir, exist_ok=True)
torch.save({"model": model.state_dict(), "config": asdict(cfg), "version": cfg.version},
           os.path.join(final_dir, "shnu_llm_v0.1.pt"))
with open(os.path.join(final_dir, "config.json"), "w") as f:
    f.write(cfg.to_json())
import shutil
os.makedirs(os.path.join(final_dir, "tokenizer"), exist_ok=True)
shutil.copy(TOK_PATH, os.path.join(final_dir, "tokenizer", "shnu_bpe.json"))
size_mb = os.path.getsize(os.path.join(final_dir, "shnu_llm_v0.1.pt"))/1e6
RUN_META["export"] = {"dir": final_dir, "weights_mb": round(size_mb,2)}
print(f"exported {size_mb:.2f} MB ->", final_dir)

## 18. Inference demo — save → reload → generate
Reload the exported model into a **fresh** object (simulating a new runtime) and generate, proving the saved artifact is self-sufficient.

In [ ]:
from tokenizers import Tokenizer
cfg_r = ShnuConfig(**json.load(open(os.path.join(final_dir, "config.json"))))
model_r = ShnuLM(cfg_r).to(DEVICE)
sd = torch.load(os.path.join(final_dir, "shnu_llm_v0.1.pt"), map_location=DEVICE, weights_only=False)["model"]
model_r.load_state_dict(sd)
tok_r = Tokenizer.from_file(os.path.join(final_dir, "tokenizer", "shnu_bpe.json"))
demo = sample_text(model_r, tok_r, cfg_r, DEVICE, prompts[0], 80, temperature=0.8, top_k=50,
                   eos_id=tok_r.token_to_id("<eos>"))
print("RELOADED MODEL OUTPUT:\n", demo)

## 19. Final report
All values below are read from this run — nothing is hard-coded.

In [ ]:
tr = RUN_META.get("train_result", {})
report = {
  "project": f"{cfg.name} {cfg.version}",
  "status": "Completed" if tr else "Not trained yet",
  "parameters": RUN_META["params"]["total"],
  "vocabulary": cfg.vocab_size,
  "context_length": cfg.block_size,
  "dataset": DATASET_INFO,
  "training_tokens": tr.get("tokens_seen"),
  "training_steps": tr.get("steps"),
  "final_train_loss": round(tr.get("final_train_loss", float('nan')), 4) if tr else None,
  "final_val_loss": round(tr.get("final_val_loss", float('nan')), 4) if tr else None,
  "perplexity": round(tr.get("final_val_ppl", float('nan')), 2) if tr else None,
  "gpu": env_report().get("gpu_name", "CPU"),
  "training_time_s": round(tr.get("train_time_s", 0), 1) if tr else None,
  "checkpoint_dir": os.path.join(cfg.out_dir, "checkpoints"),
  "model_export": RUN_META.get("export", {}).get("dir"),
}
os.makedirs(os.path.join(cfg.out_dir, "evaluation"), exist_ok=True)
json.dump({**RUN_META, "report": report},
          open(os.path.join(cfg.out_dir, "evaluation", "final_report.json"), "w"), indent=2, default=str)
print(json.dumps(report, indent=2, default=str))
print("\nSHNU-LLM v0.1 is an independently implemented and independently pretrained small model.")
print("It is NOT comparable to large frontier models; v0.1 establishes a real, scalable pipeline.")